# Cell-cell communication

Which ligands secreted by THP1 cells are likely to signal to K562 cells after LPS stimulation?

`cp.tl.c2c.infer` takes the secreted ligand-receptor pairs from CellChatDB, keeps those whose ligand genes are detected in THP1 cells and whose receptor genes are detected in K562 cells, and scores each pair by:

- its **early LR activity**: ligand expression × receptor availability over the first 45 minutes, and
- the **ligand's temporal association** in THP1 cells (tradeSeq, -log10 p).


In [ ]:
from pathlib import Path

import pandas as pd
import scanpy as sc

import chronopack as cp

DATA = Path("/new-stg/home/alex/projects/chrono_seq/analyses/thp1-k562-lps.hg38.rep")

adata = sc.read_h5ad(DATA / "output/adata_full.chk1.h5ad")
adata


## Infer ligand-receptor communication

In [ ]:
c2c = cp.tl.c2c.infer(
    adata=adata,
    lr_resource="/new-stg/home/alex/ref/cellchatdb/hs/interaction_input_CellChatDB.csv",
    association=DATA / "intmd/sc_to_tradeSeq/asso_df.both.2trajectory.nknots4.expanded.csv",
    sender="THP1",
    receiver="K562",
)
c2c.ligand_scores.head(20)


In [ ]:
c2c.write("outputs/chronopack_c2c")


## Plots

In [ ]:
fig, ax = cp.pl.c2c.interaction_scatter(
    c2c,
    receptor_detection_clip=0.2,
    p_thresh=0.05,
    x_thresh=0.1,
    max_labels=20,
    title="ChronoPack LR pair prioritization",
    xlabel="early LR activity (0-45 min)",
    ylabel="ligand -log10(p)",
)
fig.savefig("outputs/chronopack_lr_pair_scores.png", dpi=300, bbox_inches="tight")


TNF → TNFRSF1A activity against the K562 NFKBIA response, using the fitted tradeSeq curves.

In [ ]:
thp1, k562 = cp.tl.c2c.split_by_cell_type(adata, sender="THP1", receiver="K562")
tradeseq_fits = cp.tl.c2c.load_tradeseq_grid_exports(DATA / "intmd/sc_to_tradeSeq/ts_export_grid_100pts")

tnf = cp.tl.c2c.build_pair_series(
    pd.DataFrame({"ligand": ["TNF"], "receptor": ["TNFRSF1A"]}),
    sender_adata=thp1,
    receiver_adata=k562,
    response_genes=["NFKBIA"],
    prefer_fits=True,
    fits_df=tradeseq_fits["fits"],
    wmeans_df=tradeseq_fits["wmeans"],
)
fig, ax = cp.pl.c2c.plot_pair_overlay(
    tnf,
    pair=("TNF", "TNFRSF1A"),
    norm_mode="minmax",
    nfkb_onset_time=70,
    figsize=(4, 2.5),
    title="TNF->TNFRSF1A and K562 NFKBIA",
)
fig.savefig("outputs/TNF_TNFRSF1A_NFKBIA_overlay.png", dpi=300, bbox_inches="tight")


In [ ]:
fig, axes = cp.pl.c2c.plot_genes_over_time(
    adata,
    ["TNF", "TNFRSF1A", "TNFRSF1B", "NFKBIA", "TNFAIP3", "BIRC3"],
    layer="log1p",
)
fig.savefig("outputs/selected_gene_expression_over_time.png", dpi=300, bbox_inches="tight")
